In [ ]:
import numpy as np
import pandas as pd
import re

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import default_data_collator
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
from transformers import get_linear_schedule_with_warmup

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import IA3Config, get_peft_model
from peft import PeftModel, PeftConfig
from datasets import Dataset

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import gc
import time

In [ ]:
df = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv', usecols=["transaction_id","user_id","age","gender","daily_screen_time_hours","social_media_hours","gaming_hours","work_study_hours","sleep_hours","notifications_per_day","app_opens_per_day","weekend_screen_time","stress_level","academic_work_impact","addiction_level","addicted_label"])
df = df.drop(["transaction_id"], axis=1)
print(df)

columns = df.columns.values
print(columns)

# **Standard Classifiers**

In [ ]:
df = df.drop(columns=['user_id'], errors='ignore')

X = df.drop(
    columns=['addicted_label', 'addiction_level'], errors='ignore'
).reset_index(drop=True)
y = df['addicted_label'].astype(int).reset_index(drop=True)

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

models = {
    'Majority Class': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
    ),
}

sample_regimes = {
    'Few-Shot (N=40)': 40,
    'Few-Shot (N=2000)': 2000,
}

results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for regime_name, n_samples in sample_regimes.items():
  for name, model in models.items():
    accs, precs, recs, f1s = [], [], [], []

    for train_idx, test_idx in skf.split(X_encoded, y):
      X_train_full, y_train_full = (
          X_encoded.iloc[train_idx],
          y.iloc[train_idx],
      )
      X_test, y_test = X_encoded.iloc[test_idx], y.iloc[test_idx]

      if n_samples < len(X_train_full):
        X_train, _, y_train, _ = train_test_split(
            X_train_full,
            y_train_full,
            train_size=n_samples,
            stratify=y_train_full,
            random_state=42,
        )
      else:
        X_train, y_train = X_train_full, y_train_full

      model.fit(X_train, y_train)
      y_pred = model.predict(X_test)

      accs.append(accuracy_score(y_test, y_pred))
      precs.append(precision_score(y_test, y_pred, pos_label=1, zero_division=0))
      recs.append(recall_score(y_test, y_pred, pos_label=1, zero_division=0))
      f1s.append(f1_score(y_test, y_pred, pos_label=1, zero_division=0))

    results.append({
        'Regime': regime_name,
        'Model': name,
        'Accuracy': f'{np.mean(accs)*100:.2f}% (±{np.std(accs)*100:.2f})',
        'Precision': f'{np.mean(precs):.3f}',
        'Recall': f'{np.mean(recs):.3f}',
        'F1-Score': f'{np.mean(f1s):.3f}',
    })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

# **TabLLM**

# *In-Context Learning*

In [ ]:
labeled_examples = []
status = ""
pos = 500
neg = 500
index = -1
while len(labeled_examples)!=1000:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    for i in range(len(columns)-2):
      sentence += f"The {columns[i]} is {df.iloc[index,i]}. "
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
sentences = []
for m in range(len(df)):
  sentence= ""
  for i in range(len(columns)-2):
    sentence += f"The {columns[i]} is {df.iloc[m,i]}. "
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
prompt = "Classify if this person is Addicted or Not Addicted. \n\n "

i, j = 20,20
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Addicted" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Not Addicted" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, max_new_tokens=5, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print(f"Actual Answer: {df.iloc[i]['addicted_label']}")
  print("----------------------------------------------------------------------------------")

# *In-Context Learning + Text-Modification*

In [ ]:
labeled_examples = []
status = ""
pos = 1000
neg = 1000
index = -1
while len(labeled_examples)!=2000:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    sentence += f"User's ID is {df.iloc[index,0]}. "
    sentence += f"Hours on screen daily: {round(float(df.iloc[index,3])+float(df.iloc[index,4])+float(df.iloc[index,5]),2)}/24. "
    sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
    sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
    sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
    if df.iloc[index,11] == "Low":
      sentence += f"The user is not stressed. "
    elif df.iloc[index,11] == "Medium":
      sentence += f"The user is stressed. "
    else:
      sentence += f"The user is very stressed. "
    if df.iloc[index,12] == "No":
      sentence += f"The user's work is affected."
    else:
      sentence += f"The user's work is not affected."
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
sentences = []
for index in range(len(df)):
  sentence= ""
  sentence += f"User's ID is {df.iloc[index,0]}. "
  sentence += f"Hours on screen daily: {round(float(df.iloc[index,3])+float(df.iloc[index,4])+float(df.iloc[index,5]),2)}/24. "
  sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
  sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
  sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
  if df.iloc[index,11] == "Low":
    sentence += f"The user is not stressed. "
  elif df.iloc[index,11] == "Medium":
    sentence += f"The user is stressed. "
  else:
    sentence += f"The user is very stressed. "
  if df.iloc[index,12] == "No":
    sentence += f"The user's work is affected."
  else:
    sentence += f"The user's work is not affected."
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
prompt = "Classify if this person is Addicted or Not Addicted. \n\n "

i, j = 20,20
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Addicted" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Not Addicted" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, max_new_tokens=5, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print(f"Actual Answer: {df.iloc[i]['addicted_label']}")
  print("----------------------------------------------------------------------------------")

In [ ]:
prompt = "Classify if this person is Addicted or Not Addicted. \n\n "

i, j = 20,20
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Addicted" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Not Addicted" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

sample_queries = [
    f"{prompt} Description: {df.iloc[i]['Sentence']}\nVerdict:"
    for i in range(5)
]


class ComputeProfiler:

  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.latencies = []
    self.input_tokens = []
    self.output_tokens = []
    self.peak_vrams = []

  def profile_query(self, query_fn, *args, **kwargs):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    output_text, input_prompt = query_fn(*args, **kwargs)
    end_time = time.perf_counter()

    self.latencies.append(end_time - start_time)
    self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

    self.input_tokens.append(len(self.tokenizer.encode(input_prompt)))
    self.output_tokens.append(len(self.tokenizer.encode(output_text)))

    return output_text

  def summary(self, model_name: str, method_name: str):
    return {
        "Paradigm / Method": method_name,
        "Model": model_name,
        "Avg Latency (s)": (
            f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}"
        ),
        "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
        "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
        "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
    }

profiler = ComputeProfiler(generator.tokenizer)


def run_qwen_inference(q):
  pad_id = (
      generator.tokenizer.pad_token_id
      if generator.tokenizer.pad_token_id is not None
      else generator.tokenizer.eos_token_id
  )

  res = generator(
      q,
      max_new_tokens=64,
      return_full_text=False,
      pad_token_id=pad_id,
      do_sample=False,
  )

  generated_entry = res[0]["generated_text"]

  if isinstance(generated_entry, list):
    out_text = generated_entry[-1]["content"].strip()
  else:
    out_text = str(generated_entry).strip()

  return out_text, q

for s in sample_queries:
  profiler.profile_query(run_qwen_inference, s)

print(profiler.summary("Qwen2.5-3B-Instruct (4-bit)", "ICL"))

#*Markdown*

In [ ]:
df = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv', usecols=["transaction_id","user_id","age","gender","daily_screen_time_hours","social_media_hours","gaming_hours","work_study_hours","sleep_hours","notifications_per_day","app_opens_per_day","weekend_screen_time","stress_level","academic_work_impact","addiction_level","addicted_label"])
df["daily_screen_time_hours"] = df["daily_screen_time_hours"] + df["social_media_hours"] + df["gaming_hours"]
df = df.drop(["transaction_id","age","gender","social_media_hours","gaming_hours","notifications_per_day","app_opens_per_day"], axis=1)
print(df)

columns = df.columns.values
print(columns)

In [ ]:
sentences = []
for index in range(len(df)):
  sentence= ""
  sentence += f"User's ID is {df.iloc[index,0]}. "
  sentence += f"Hours on screen daily: {df.iloc[index,1]}/24. "
  sentence += f"Hours working daily: {df.iloc[index,2]}/24. "
  sentence += f"Hours sleeping daily: {df.iloc[index,3]}/24. "
  sentence += f"Hours on phone on a weekend: {df.iloc[index,4]}/24. "
  if df.iloc[index,5] == "Low":
    sentence += f"The user is not stressed. "
  elif df.iloc[index,5] == "Medium":
    sentence += f"The user is stressed. "
  else:
    sentence += f"The user is very stressed. "
  if df.iloc[index,6] == "No":
    sentence += f"The user's work is affected."
  else:
    sentence += f"The user's work is not affected."
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
df_addicted_mini = df[df['addicted_label'] == 1].head(20)
df_nonaddicted_mini = df[df['addicted_label'] == 0].head(20)
merged_df = pd.concat([df_addicted_mini, df_nonaddicted_mini])
drop_indexes = merged_df.index
df_copy = df
df_copy.drop(drop_indexes, inplace=True)
df_copy = df_copy.reset_index(drop=True)

print(df_copy)

merged_df = merged_df.drop(['addiction_level','addicted_label','Sentence'], axis=1)

table_str = merged_df.to_markdown()

print(table_str)

In [ ]:
prompt = "Classify if this person is Addicted or Not Addicted. \n\n"

prompt += table_str
print(prompt)

for i in range(100):
  test_sentence = df_copy.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, max_new_tokens=5, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print(f"Actual Answer: {df.iloc[i]['addicted_label']}")
  print("----------------------------------------------------------------------------------")

#*Question-Answering*

In [ ]:
labeled_examples = []
status = ""
pos = 20
neg = 20
index = -1
while len(labeled_examples)!=40:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    sentence += f"User's ID is {df.iloc[index,0]}. "
    sentence += f"Age is {df.iloc[index,1]}. "
    sentence += f"Gender is {df.iloc[index,2]}. "
    sentence += f"Hours on screen daily: {df.iloc[index,3]}/24. "
    sentence += f"Hours on social media daily: {df.iloc[index,4]}/24. "
    sentence += f"Hours gaming daily: {df.iloc[index,5]}/24. "
    #sentence += f"Hours on screen daily: {round(float(df.iloc[index,3])+float(df.iloc[index,4])+float(df.iloc[index,5]),2)}/24. "
    sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
    sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
    sentence += f"Notifications per day: {df.iloc[index,8]}. "
    sentence += f"App opens per day: {df.iloc[index,9]}. "
    sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
    if df.iloc[index,11] == "Low":
      sentence += f"The user is not stressed. "
    elif df.iloc[index,11] == "Medium":
      sentence += f"The user is stressed. "
    else:
      sentence += f"The user is very stressed. "
    if df.iloc[index,12] == "No":
      sentence += f"The user's work is affected."
    else:
      sentence += f"The user's work is not affected."
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
prompt = ""
i, j = 20,20
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Addicted" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Not Addicted" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

print(prompt)

In [ ]:
final_prompt = prompt + f"Give me the IDs of all the Addicted people."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the ID of user with the smallest Age."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the ID of the user with the highest hours on screen daily as well as what those hours were."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the IDs of all users where Gender is Female and their hours sleeping daily."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the IDs of all users who are stressed or very stressed."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the IDs of all users whose work is affected."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the IDs of all users who receive more than 150 notifications per day."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Give me the IDs of all users who spent more than 3 hours sleeping and gaming."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

In [ ]:
print(f"Number of addicted men: {sum(df[df["gender"] == "Male"]["addicted_label"].tolist())}")
print(f"Number of addicted women: {sum(df[df["gender"] == "Female"]["addicted_label"].tolist())}")
print(f"Number of addicted others: {sum(df[df["gender"] == "Other"]["addicted_label"].tolist())}")
final_prompt = prompt + f"Which gender is more prone to phone addiction?"
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
df_copy = df[df["addicted_label"]==1]
print(f"Average daily screen time for an addicted person: {sum(df_copy['daily_screen_time_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, what is the daily screen time of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average social media hours for an addicted person: {sum(df_copy['social_media_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the daily social media hours of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average gaming hours for an addicted person: {sum(df_copy['gaming_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the daily gaming hours of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average work/study hours for an addicted person: {sum(df_copy['work_study_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the daily work or study hours of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average sleep hours for an addicted person: {sum(df_copy['sleep_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the sleep hours of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average notifications per day for an addicted person: {sum(df_copy['notifications_per_day'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the notifications per day of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average app opens per day for an addicted person: {sum(df_copy['app_opens_per_day'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, how many are the app opens per day of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average weekend screen time for an addicted person: {sum(df_copy['weekend_screen_time'].tolist())/len(df_copy)}")
final_prompt = prompt + f"On average, what is the weekend screen time of an addicted person? Just give me a number."
answer = generator(final_prompt, return_full_text=False)
print(answer[0]['generated_text'])

 *Chain of Thought*

In [ ]:
final_prompt = prompt + f"Iterate through the provided rows and identify every user who is labeled as 'Addicted', then compile a list of all their IDs. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and keep track of all the users' ages, then in the end return the ID of the one with the smallest age. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and keep track of all the users' daily screen hours, then return the ID of the user with the most hours and how many those hours were. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and identify all the users labeled as 'Female', then return their IDs and daily sleeping hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and identify all the users labeled as 'stressed' or 'very stressed', then return their IDs. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and identify all the users whose work is said to be affected, then return their IDs. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and identify all the users who receive more than 150 daily notifications, then return their IDs. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

final_prompt = prompt + f"Iterate through the provided rows and identify all the users who spent more than three hours sleeping, then out of those users identify those that spend more than three hours gaming, then return their IDs. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=3048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")

In [ ]:
print(f"Number of addicted men: {sum(df[df["gender"] == "Male"]["addicted_label"].tolist())}")
print(f"Number of addicted women: {sum(df[df["gender"] == "Female"]["addicted_label"].tolist())}")
print(f"Number of addicted others: {sum(df[df["gender"] == "Other"]["addicted_label"].tolist())}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users of each gender which are labeled as addicted, then tell me which out of them displays the most instances of addiction. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
df_copy = df[df["addicted_label"]==1]
print(f"Average daily screen time for an addicted person: {sum(df_copy['daily_screen_time_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily screen time hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average social media hours for an addicted person: {sum(df_copy['social_media_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily social media hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average gaming hours for an addicted person: {sum(df_copy['gaming_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily gaming hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average work/study hours for an addicted person: {sum(df_copy['work_study_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily work hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average sleep hours for an addicted person: {sum(df_copy['sleep_hours'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily sleep hours. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average notifications per day for an addicted person: {sum(df_copy['notifications_per_day'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily notifications. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average app opens per day for an addicted person: {sum(df_copy['app_opens_per_day'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their daily app opens. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])
print("------------------------------------------------------------------------")
print(f"Average weekend screen time for an addicted person: {sum(df_copy['weekend_screen_time'].tolist())/len(df_copy)}")
final_prompt = prompt + f"Iterate through all the rows and identify all the users who are labeled as addicted, then find the average of their weekend screen time. Explain your through process every step of the way and do not produce code."
answer = generator(final_prompt, max_new_tokens=2048, return_full_text=False)
print(answer[0]['generated_text'])

# *T-Few*

In [ ]:
labeled_examples = []
status = ""
pos = 1000
neg = 1000
index = -1
while len(labeled_examples)!=2000:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    sentence += f"Hours on screen daily: {df.iloc[index,3]+df.iloc[index,4]+df.iloc[index,5]}/24. "
    sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
    sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
    sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
    if df.iloc[index,11] == "Low":
      sentence += f"The user is not stressed. "
    elif df.iloc[index,11] == "Medium":
      sentence += f"The user is stressed. "
    else:
      sentence += f"The user is very stressed. "
    if df.iloc[index,12] == "No":
      sentence += f"The user's work is affected."
    else:
      sentence += f"The user's work is not affected."
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
sentences = []
for index in range(len(df)):
  sentence= ""
  sentence += f"Hours on screen daily: {df.iloc[index,3]+df.iloc[index,4]+df.iloc[index,5]}/24. "
  sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
  sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
  sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
  if df.iloc[index,11] == "Low":
    sentence += f"The user is not stressed. "
  elif df.iloc[index,11] == "Medium":
    sentence += f"The user is stressed. "
  else:
    sentence += f"The user is very stressed. "
  if df.iloc[index,12] == "No":
    sentence += f"The user's work is affected."
  else:
    sentence += f"The user's work is not affected."
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)
train_dataset = Dataset.from_pandas(labeled_df)

text_column = "Sentence"
label_column = "Label"

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(examples[label_column], max_length=5, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

column_names = train_dataset.column_names
tokenized_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=column_names)

split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_df = split_dataset["train"]
eval_df = split_dataset["test"]

train_dataloader = DataLoader(train_df, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True)
eval_dataloader = DataLoader(eval_df, collate_fn=default_data_collator, batch_size=8, pin_memory=True)

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
num_epochs = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-2)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

In [ ]:
for i in range(100):
  input = unlabeled_df.iloc[i].values[15] + f" Is this person Addicted or Not Addicted?"
  print(input)
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=10)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)
      print(f"Actual Answer: {unlabeled_df.iloc[i].values[14]}")

In [ ]:
class ComputeProfiler:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.latencies = []
        self.input_tokens = []
        self.output_tokens = []
        self.peak_vrams = []

    def profile_query(self, query_fn, *args, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        start_time = time.perf_counter()
        output_text, input_prompt = query_fn(*args, **kwargs)
        end_time = time.perf_counter()

        self.latencies.append(end_time - start_time)
        self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

        self.input_tokens.append(len(self.tokenizer.encode(input_prompt)))
        self.output_tokens.append(len(self.tokenizer.encode(output_text)))

        return output_text

    def summary(self, model_name: str, method_name: str):
        return {
            "Paradigm / Method": method_name,
            "Model": model_name,
            "Avg Latency (s)": f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}",
            "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
            "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
            "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
        }

profiler = ComputeProfiler(tokenizer)

def run_mt0_inference(row_idx):
    prompt_str = unlabeled_df.iloc[row_idx].values[15] + " Is this person Addicted or Not Addicted?"
    inputs = tokenizer(prompt_str, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_new_tokens=10
        )
        prediction = tokenizer.batch_decode(outputs.detach().cpu(), skip_special_tokens=True)[0].strip()

    return prediction, prompt_str

for i in range(5):
    profiler.profile_query(run_mt0_inference, i)

print(profiler.summary("mt0-large (FP16 / T-Few)", "T-Few / (IA)^3"))

In [ ]:
input = "On average, how many hours does an addicted person spend working daily?"
print(input)
input = tokenizer(input, return_tensors="pt").to(device)

with torch.no_grad():
  input = {k: v.to(device) for k, v in input.items()}
  outputs = model.generate(input_ids=input["input_ids"])
  prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
  print(prediction)

# *T-Few 2*

In [ ]:
labeled_examples = []
status = ""
pos = 1000
neg = 1000
index = -1
while len(labeled_examples)!=2000:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    sentence += f"Hours on screen daily: {df.iloc[index,3]+df.iloc[index,4]+df.iloc[index,5]}/24. "
    sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
    sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
    sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
    if df.iloc[index,11] == "Low":
      sentence += f"The user is not stressed. "
    elif df.iloc[index,11] == "Medium":
      sentence += f"The user is stressed. "
    else:
      sentence += f"The user is very stressed. "
    if df.iloc[index,12] == "No":
      sentence += f"The user's work is affected."
    else:
      sentence += f"The user's work is not affected."
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
sentences = []
for index in range(len(df)):
  sentence= ""
  sentence += f"Hours on screen daily: {df.iloc[index,3]+df.iloc[index,4]+df.iloc[index,5]}/24. "
  sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
  sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
  sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
  if df.iloc[index,11] == "Low":
    sentence += f"The user is not stressed. "
  elif df.iloc[index,11] == "Medium":
    sentence += f"The user is stressed. "
  else:
    sentence += f"The user is very stressed. "
  if df.iloc[index,12] == "No":
    sentence += f"The user's work is affected."
  else:
    sentence += f"The user's work is not affected."
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)

text_column = "Sentence"
label_column = "Label"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    prompt = f"Data: {examples[text_column]} Verdict:"
    answer = f" {examples[label_column]}"

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    input_ids = prompt_ids + answer_ids

    labels = [-100] * len(prompt_ids) + answer_ids

    padding_len = 128 - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding_len
    labels += [-100] * padding_len

    return {
        "input_ids": torch.tensor(input_ids[:128]),
        "labels": torch.tensor(labels[:128]),
        "attention_mask": torch.tensor(([1] * len(input_ids[:128])))
    }

full_dataset = Dataset.from_pandas(labeled_df)
split_dataset = full_dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

tokenized_train = train_dataset.map(preprocess_function, batched=False)
tokenized_eval = eval_dataset.map(preprocess_function, batched=False)

tokenized_train = tokenized_train.remove_columns(["Sentence", "Label"])
tokenized_eval = tokenized_eval.remove_columns(["Sentence", "Label"])

train_dataloader = DataLoader(tokenized_train, shuffle=True, collate_fn=default_data_collator, batch_size=8)
eval_dataloader = DataLoader(tokenized_eval, collate_fn=default_data_collator, batch_size=8)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

peft_config = IA3Config(task_type="CAUSAL_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
num_epochs = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-2)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

In [ ]:
for i in range(100):
  input = unlabeled_df.iloc[i].values[15] + f" Is this person Addicted or Not Addicted?"
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=20)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)
      print(f"Actual Answer: {unlabeled_df.iloc[i].values[14]}")

In [ ]:
input = "Based on your data, what is the average daily screentime for an addicted user?"
print(input)
input = tokenizer(input, return_tensors="pt").to(device)

with torch.no_grad():
  input = {k: v.to(device) for k, v in input.items()}
  outputs = model.generate(input_ids=input["input_ids"])
  prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
  print(prediction)

# *PAL*

In [ ]:
sentences = []
for index in range(len(df)):
  sentence= ""
  sentence += f"User's ID is {df.iloc[index,0]}. "
  sentence += f"Age is {df.iloc[index,1]}. "
  sentence += f"Gender is {df.iloc[index,2]}. "
  sentence += f"Hours on screen daily: {df.iloc[index,3]}/24. "
  sentence += f"Hours on social media daily: {df.iloc[index,4]}/24. "
  sentence += f"Hours gaming daily: {df.iloc[index,5]}/24. "
  sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
  sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
  sentence += f"Notifications per day: {df.iloc[index,8]}. "
  sentence += f"App opens per day: {df.iloc[index,9]}. "
  sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
  if df.iloc[index,11] == "Low":
    sentence += f"The user is not stressed. "
  elif df.iloc[index,11] == "Medium":
    sentence += f"The user is stressed. "
  else:
    sentence += f"The user is very stressed. "
  if df.iloc[index,12] == "No":
    sentence += f"The user's work is affected."
  else:
    sentence += f"The user's work is not affected."
  sentences.append(sentence)

df['Sentence'] = sentences
print(df)

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find which gender has a higher number of addicted users.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the average daily screen time for an addicted user.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the average social media for an addicted user.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the ID of the user with the smallest age.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")


prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the ID of the user with the highest daily screen hours.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")


prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the IDs of all Female users who sleep for less than seven hours.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")


prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find the IDs of all users who sleep for more than three hours and also spend more than three hours gaming.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find if users who spend more time on social media or users who spend more time gaming are more prone to sleeping for less than 6 hours.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find which age group (ex. 10-20) spends the fewest hours sleeping on average.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find whether users who spend more time on social media or users who work more exhibit more instances of High stress.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

prompt = f"""My dataframe is called df and it possesses the following columns {columns}.
            Using only pandas and numpy libraries give me code to find whether addicted or not addicted users exhibit higher weekend screen time on average.
            Don't give me any text whatsoever aside from code."""
answer = generator(prompt, temperature=0.1, return_full_text=False)[0]['generated_text'].strip()
print(f"Prompt: {prompt}")
print(f"AI Proposed Code: {answer}")
print("------------------------------------------------------------------------")

 *ReAct*

In [ ]:
#q = "I want to find if users who spend more time on social media or users who spend more time gaming are more prone to sleeping for less than 6 hours."
#q = "I want to find which age group (ex. 10-20) spends the fewest hours sleeping on average."
q = "I want to find whether users who have higher social media hours or users who have higher work/study hours exhibit more instances of their stress level being High."
#q = "I want to find whether addicted or not addicted users exhibit higher weekend screen time on average."

thought_prompt = f"""My dataframe is called df and it possesses the following columns {list(col)}.\n
                  Task: {q}\n
                  Do NOT write code, just break down the step by step filtering process for how you would achieve this."""

thought_answer = generator(
    thought_prompt,
    temperature=0.1,
    max_new_tokens=512,
    return_full_text=False,
    pad_token_id=pad_id,
    do_sample=False
)[0]['generated_text'].strip()

skip_first = 1
observation_answer = "FAILURE"
action_answer = ""
current_obs = ""
max_retries = 3

while "FAILURE" in observation_answer and max_retries > 0:
    max_retries -= 1

    action_prompt = f"""My dataframe is called df and it possesses the following columns {list(col)}.\n
                    Task: {q}\n
                    Follow the steps of the Thought to craft code. Assign your final computed answer to a variable called `result`.
                    Do not output anything other than pure python code wrapped in ```python ... ```.
                    Thought: {thought_answer}\n"""

    if "FAILURE" in observation_answer and skip_first == 0:
        action_prompt += f"""Your last attempt contained a mistake. Review the feedback and fix the code:
                          Correction Feedback: {observation_answer}"""

    action_answer = generator(
        action_prompt,
        temperature=0.1,
        max_new_tokens=512,
        return_full_text=False,
        pad_token_id=pad_id,
        do_sample=False
    )[0]['generated_text'].strip()
    skip_first = 0

    current_obs = execute_safely(action_answer, df)

    observation_prompt = f"""You are a senior data scientist who must be logically rigorous and avoid doubting correct code. My dataframe is called df with columns {list(col)}.\n
                        Task: {q}\n
                        Code: {action_answer}\n
                        Execution Output: {current_obs}\n
                        Examine if the code executes cleanly and achieves the goal without errors.
                        If it succeeded and gave the correct answer, respond strictly with 'SUCCESS'.
                        If there is a runtime error or logic error, respond 'FAILURE' followed by a short explanation."""

    if str(current_obs).startswith("Error:"):
        observation_answer = f"FAILURE: Code crashed with {current_obs}"
    else:
        observation_answer = generator(
            observation_prompt,
            temperature=0.1,
            max_new_tokens=256,
            return_full_text=False,
            pad_token_id=pad_id,
            do_sample=False
        )[0]['generated_text'].strip()

print(f"Dataset {i} | Q: {q}")
print(f"Proposed Code:\n{action_answer}")
print(f"Verifier Verdict: {'SUCCESS' if 'SUCCESS' in observation_answer else 'EXHAUSTED RETRIES'}")
print(f"Execution Result: {current_obs}")
print(f"Actual Answer:    {actual_ans}")

In [ ]:
class ComputeProfiler:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.latencies = []
        self.input_tokens = []
        self.output_tokens = []
        self.peak_vrams = []

    def profile_query(self, query_fn, *args, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        start_time = time.perf_counter()
        final_code, all_inputs, all_outputs = query_fn(*args, **kwargs)
        end_time = time.perf_counter()

        self.latencies.append(end_time - start_time)
        self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

        total_in = sum(len(self.tokenizer.encode(p)) for p in all_inputs)
        total_out = sum(len(self.tokenizer.encode(g)) for g in all_outputs)
        self.input_tokens.append(total_in)
        self.output_tokens.append(total_out)

        return final_code

    def summary(self, model_name: str, method_name: str):
        return {
            "Paradigm / Method": method_name,
            "Model": model_name,
            "Avg Latency (s)": f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}",
            "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
            "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
            "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
        }

profiler = ComputeProfiler(generator.tokenizer)

sample_react_tasks = [
    "I want to find whether addicted or not addicted users exhibit higher weekend screen time on average.",
    "I want to find which age group spends the fewest hours sleeping on average.",
    "Calculate the maximum 'Daily Screen Time' grouped by 'Gender'.",
    "Compute the median 'Daily Social Media Hours' across all users.",
    "Find the standard deviation of 'Age' grouped by 'Addicted Label'"
]

def run_react_single(q, current_columns, target_df):
    all_inputs = []
    all_outputs = []
    pad_id = generator.tokenizer.pad_token_id or generator.tokenizer.eos_token_id

    # 1. Thought Stage
    thought_prompt = f"""My dataframe is called df and it possesses the following columns {list(current_columns)}.\n
                      Task: {q}\n
                      Do NOT write code, just break down the step by step filtering process for how you would achieve this."""
    all_inputs.append(thought_prompt)

    thought_answer = generator(
        thought_prompt,
        temperature=0.1,
        max_new_tokens=512,
        return_full_text=False,
        pad_token_id=pad_id,
        do_sample=False
    )[0]['generated_text'].strip()
    all_outputs.append(thought_answer)

    skip_first = 1
    observation_answer = "FAILURE"
    action_answer = ""
    current_obs = ""
    max_retries = 3

    while "FAILURE" in observation_answer and max_retries > 0:
        max_retries -= 1

        action_prompt = f"""My dataframe is called df and it possesses the following columns {list(current_columns)}.\n
                          Task: {q}\n
                          Follow the steps of the Thought to craft code. Assign your final computed answer to a variable called `result`.
                          Do not output anything other than pure python code wrapped in ```python ... ```.
                          Thought: {thought_answer}\n"""

        if "FAILURE" in observation_answer and skip_first == 0:
            action_prompt += f"""Your last attempt contained a mistake. Review the feedback and fix the code:
                              Correction Feedback: {observation_answer}"""

        all_inputs.append(action_prompt)
        action_answer = generator(
            action_prompt,
            temperature=0.1,
            max_new_tokens=512,
            return_full_text=False,
            pad_token_id=pad_id,
            do_sample=False
        )[0]['generated_text'].strip()
        all_outputs.append(action_answer)
        skip_first = 0

        current_obs = execute_safely(action_answer, target_df)

        observation_prompt = f"""You are a senior data scientist who must be logically rigorous and avoid doubting correct code. My dataframe is called df with columns {list(current_columns)}.\n
                              Task: {q}\n
                              Code: {action_answer}\n
                              Execution Output: {current_obs}\n
                              Examine if the code executes cleanly and achieves the goal without errors.
                              If it succeeded and gave the correct answer, respond strictly with 'SUCCESS'.
                              If there is a runtime error or logic error, respond 'FAILURE' followed by a short explanation."""

        if str(current_obs).startswith("Error:"):
            observation_answer = f"FAILURE: Code crashed with {current_obs}"
        else:
            all_inputs.append(observation_prompt)
            observation_answer = generator(
                observation_prompt,
                temperature=0.1,
                max_new_tokens=256,
                return_full_text=False,
                pad_token_id=pad_id,
                do_sample=False
            )[0]['generated_text'].strip()
            all_outputs.append(observation_answer)

    return action_answer, all_inputs, all_outputs

target_df = df if 'df' in globals() else pd.DataFrame()
cols_to_pass = list(target_df.columns) if not target_df.empty else ['User ID', 'Age', 'Daily Screen Time', 'Addicted Label']

for t in sample_react_tasks:
    profiler.profile_query(run_react_single, t, cols_to_pass, target_df)

print("\n" + "="*50)
print(profiler.summary("Qwen2.5-3B-Instruct (4-bit)", "ReAct"))
print("="*50)